In [0]:
from pyspark.sql import functions as F

train_table = "workspace.default.train_00000_of_00001"
eval_table = "workspace.default.eval_00000_of_00001"
bronze_table = "workspace.default.bronze_rbi_circular_qa"

train_df = (
    spark.table(train_table)
    .withColumn("split", F.lit("train"))
    .withColumn("source_table", F.lit(train_table))
)

eval_df = (
    spark.table(eval_table)
    .withColumn("split", F.lit("eval"))
    .withColumn("source_table", F.lit(eval_table))
)

bronze_df = (
    train_df.unionByName(eval_df, allowMissingColumns=True)
    .withColumn("ingested_at", F.current_timestamp())
)

(
    bronze_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

display(spark.table(bronze_table).limit(20))
print("Bronze table created:", bronze_table)
print("Bronze rows:", spark.table(bronze_table).count())


document,filename,model_name,regulation_area,applicable_to,issued_on,key_topics,chunks_text,is_table,question,answer,evaluation_criteria,category,estimated_difficulty,rephrased_question,rephrased_answer,data_source,split,source_table,ingested_at
RBI_2019-2020_228DOR.BP.BC.No.68_21.04.018_2019-20_2020-04-30,RBI_2019-2020_228DOR.BP.BC.No.68_21.04.018_2019-20_2020-04-30_text_part1.txt,models/gemini-2.0-flash,Regulation of Banks and Financial Institutions,"All Scheduled Commercial Banks, Payments Banks, Local Area Banks, All India Financial Institutions, All Co-operative Banks",2020-04-29,"List(Extension of timelines for regulatory returns, Submission of regulatory returns, COVID-19 pandemic related relaxations, Statutory returns)","![](_page_0_Picture_0.jpeg) ## भारतीय �रजवर्ब�क **RESERVE BANK OF INDIA www.rbi.org.in** RBI/2019-20/228 DOR.BP.BC.No.68/21.04.018/2019-20 April 29, 2020 All Scheduled Commercial Banks (including RRBs and Small Finance Banks), Payments Banks and Local Area Banks, All India Financial Institutions, All Co-operative Banks, Madam / Dear Sir, ## **Submission of regulatory returns - Extension of timelines** In order to mitigate the difficulties in timely submission of various regulatory returns, in view of disruptions on account of COVID-19 pandemic, it has been decided to extend the timelines for their submission. 2. Accordingly, all regulatory returns required to be submitted by the above entities to the Department of Regulation can be submitted with a delay of upto 30 days from the due date. The extension will be applicable to regulatory returns required to be submitted upto June 30, 2020. Further details are furnished in the [Annex.](#page-1-0) Those entities that are in a position to submit the returns earlier may continue to do so. 3. It may be noted that no extension in timeline is permitted for submission of statutory returns i.e. returns prescribed under the Banking Regulation Act, 1949, RBI Act, 1934 or any other Act (for instance, returns related to CRR/SLR). 4. Further, all communication to the Department of Regulation should be through corporate e-mail to the extent possible (i.e., without involving physical movement of papers). This arrangement shall continue till further notice. Yours faithfully, (Saurav Sinha) Chief General Manager-in-Charge > िविनयमन िवभाग,क��ीय कायार्लय, 12 व� और 13 व� मंिजल, क��ीय कायार्लय भवन, शहीद भगत �संह मागर्,फोटर्,मुंबई-400001 दूरभाष: 022-22601000 फै क्स: 022-22705691 ई-मेल: cgmicdor@rbi.org.in \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_ Department of Regulation, Central Office, 12th and 13th Floor, Central Office Building, Shahid Bhagat Singh Marg, Fort, Mumbai- 400 001 Tel: 022- 2260 1000 Fax: 022-2270 5691 email: cgmicdor@rbi.org.in ![](_page_0_Picture_15.jpeg) ## **Annex: List of regulatory returns which can be submitted with a delay of a maximum of 30 days from the due date**",false,What relaxations were provided by the Reserve Bank of India regarding the submission of regulatory returns in light of the COVID-19 pandemic?,"Due to disruptions caused by the COVID-19 pandemic, the Reserve Bank of India allowed regulated entities to submit regulatory returns with a delay of up to 30 days from the original due date. This extension was applicable to returns required to be submitted up to June 30, 2020.","The answer should accurately state the reason for the extension, the length of the extension, and the period for which it was applicable.",fact-based,3,,,original,train,workspace.default.train_00000_of_00001,2026-04-18T06:18:17.147Z
RBI_2019-2020_228DOR.BP.BC.No.68_21.04.018_2019-20_2020-04-30,RBI_2019-2020_228DOR.BP.BC.No.68_21.04.018_2019-20_2020-04-30_text_part1.txt,models/gemini-2.0-flash,Regulation of Banks and Financial Institutions,"All S

Bronze table created: workspace.default.bronze_rbi_circular_qa
Bronze rows: 48934


In [0]:
from pyspark.sql import functions as F

bronze_table = "workspace.default.bronze_rbi_circular_qa"
silver_table = "workspace.default.silver_rbi_circular_qa"
gold_chunks_table = "workspace.default.gold_rbi_circular_chunks"
gold_eval_table = "workspace.default.gold_rbi_eval"

raw_df = spark.table(bronze_table)

clean_df = (
    raw_df
    .withColumn("document", F.trim(F.col("document")))
    .withColumn("filename", F.trim(F.col("filename")))
    .withColumn("regulation_area", F.trim(F.col("regulation_area")))
    .withColumn("applicable_to", F.trim(F.col("applicable_to")))
    .withColumn("issued_on", F.expr("try_cast(issued_on as date)"))
    .withColumn("chunks_text", F.regexp_replace(F.col("chunks_text"), r"\s+", " "))
    .withColumn("question", F.regexp_replace(F.col("question"), r"\s+", " "))
    .withColumn("answer", F.regexp_replace(F.col("answer"), r"\s+", " "))
    .withColumn(
        "qa_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("split"), F.lit("")),
                F.coalesce(F.col("document"), F.lit("")),
                F.coalesce(F.col("filename"), F.lit("")),
                F.coalesce(F.col("question"), F.lit("")),
                F.coalesce(F.col("answer"), F.lit("")),
            ),
            256,
        ),
    )
)

(
    clean_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

chunks_df = (
    clean_df
    .select(
        "document",
        "filename",
        "regulation_area",
        "applicable_to",
        "issued_on",
        "key_topics",
        "chunks_text",
        "is_table",
    )
    .dropna(subset=["document", "filename", "chunks_text"])
    .dropDuplicates(["document", "filename", "chunks_text"])
    .withColumn(
        "chunk_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("document"), F.lit("")),
                F.coalesce(F.col("filename"), F.lit("")),
                F.coalesce(F.col("chunks_text"), F.lit("")),
            ),
            256,
        ),
    )
    .withColumn(
        "retrieval_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("Document: "), F.coalesce(F.col("document"), F.lit(""))),
            F.concat(F.lit("Filename: "), F.coalesce(F.col("filename"), F.lit(""))),
            F.concat(F.lit("Regulation area: "), F.coalesce(F.col("regulation_area"), F.lit(""))),
            F.concat(F.lit("Applicable to: "), F.coalesce(F.col("applicable_to"), F.lit(""))),
            F.concat(F.lit("Issued on: "), F.coalesce(F.col("issued_on").cast("string"), F.lit(""))),
            F.concat(F.lit("Chunk: "), F.coalesce(F.col("chunks_text"), F.lit(""))),
        ),
    )
)

(
    chunks_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_chunks_table)
)

eval_df = (
    clean_df
    .filter(F.col("split") == "eval")
    .select(
        "qa_id",
        "document",
        "question",
        "answer",
        "rephrased_question",
        "rephrased_answer",
        "evaluation_criteria",
        "category",
        "estimated_difficulty",
    )
)

(
    eval_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_eval_table)
)

print("Silver rows:", spark.table(silver_table).count())
print("Gold chunk rows:", spark.table(gold_chunks_table).count())
print("Gold eval rows:", spark.table(gold_eval_table).count())

display(spark.table(gold_chunks_table).limit(20))


Silver rows: 48934
Gold chunk rows: 756
Gold eval rows: 1000


document filename regulation_area applicable_to issued_on key_topics chunks_text is_table chunk_id retrieval_text RBI_2019-2020_132FMRD.FMID No.23_02.05.002_2019-20_2020-01-01 RBI_2019-2020_132FMRD.FMID No.23_02.05.002_2019-20_2020-01-01_text_part1.txt Currency Derivatives All Category-I Authorised Dealer Banks 2020-01-01 List(Reporting of OTC Currency Derivative transactions, Trade Repository (TR) reporting, Client transactions in currency derivatives, Threshold for reporting) ![](_page_0_Picture_0.jpeg) ## RESERVE BANK OF INDIA www.rbi.org.in RBI/2019-20/132 FMRD.FMID No.23/02.05.002/2019-20 January 01, 2020 All Category-I Authorised Dealer Banks Madam/Sir, ## **Reporting of OTC Currency Derivative transactions to trade repository** Please refer to our [circular FMD.MSRG.No.94/02.05.002/2013-14 dated December 04, 2013](https://www.rbi.org.in/Scripts/NotificationUser.aspx?Id=8619&Mode=0) on the captioned subject, wherein a threshold of USD 1 million, and equivalent thereof in other currencies, was stipulated for reporting client transactions in currency derivatives (currency swaps and FCY FRA/IRS) to the Trade Repository (TR). 2. It has now been decided that all client transactions in currency derivatives, including those with notional amount of below USD 1 mn, shall now be reported to the TR, with effect from January 06, 2020. 3. As a one-time measure, in order to update the transactions in the Trade Repository, AD Category – I banks shall report all outstanding client transactions with notional amount below USD 1 mn to the TR by January 31, 2020. 4. These directions are issued under section 45W of RBI Act and shall come into force with effect from the date of these directions. Yours faithfully (Manoj Kumar) Deputy General Manager false 792385fd58b28a1eec46a18e5bf76288bfdc54a6f0f3097440872775914d228b Document: RBI_2019-2020_132FMRD.FMID No.23_02.05.002_2019-20_2020-01-01
Filename: RBI_2019-2020_132FMRD.FMID No.23_02.05.002_2019-20_2020-01-01_text_part1.txt
Regulation area: Currency Derivatives
Applicable to: All Category-I Authorised Dealer Banks
Issued on: 2020-01-01
Chunk: ![](_page_0_Picture_0.jpeg) ## RESERVE BANK OF INDIA www.rbi.org.in RBI/2019-20/132 FMRD.FMID No.23/02.05.002/2019-20 January 01, 2020 All Category-I Authorised Dealer Banks Madam/Sir, ## **Reporting of OTC Currency Derivative transactions to trade repository** Please refer to our [circular FMD.MSRG.No.94/02.05.002/2013-14 dated December 04, 2013](https://www.rbi.org.in/Scripts/NotificationUser.aspx?Id=8619&Mode=0) on the captioned subject, wherein a threshold of USD 1 million, and equivalent thereof in other currencies, was stipulated for reporting client transactions in currency derivatives (currency swaps and FCY FRA/IRS) to the Trade Repository (TR). 2. It has now been decided that all client transactions in currency derivatives, including those with notional amount of below USD 1 mn, shall now be reported to the TR, with effect from January 06, 2020. 3. As a one-time measure, in order to update the transactions in the Trade Repository, AD Category – I banks shall report all outstanding client transactions with notional amount below USD 1 mn to the TR by January 31, 2020. 4. These directions are issued under section 45W of RBI Act and shall come into force with effect from the date of these directions. Yours faithfully (Manoj Kumar) Deputy General Manager RBI_2019-2020_134A.P. (DIR Series) Circular No. 14_2020-01-02 RBI_2019-2020_134A.P. (DIR Series) Circular No. 14_2020-01-02_text_part1.txt Foreign Exchange Management Act (FEMA) All Category – I Authorised Dealer Banks 2020-01-01 List(Exim Bank's Government of India supported Line of Credit (LOC) to Banco Exterior De Cuba, Financing installation of 75 MW Photovoltaic Solar Parks in the Republic of Cuba, Shipments under the LoC shall be declared in Export Declaration Form, Payment of commission in free foreign exchange) ![](_page_0_Picture_0.jpeg) RBI/2019-20/134 A.P. (DIR Series) Circular No. 14 Jan

In [0]:
%pip install -U databricks-vectorsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-91ff250e-8342-4900-8b7c-77392dda30ae
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-91ff250e-8342-4900-8b7c-77392dda30ae
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-91ff250e-8342-4900-

In [0]:
dbutils.library.restartPython()


In [0]:
from databricks.vector_search.client import VectorSearchClient
print("Vector Search import works")

Vector Search import works


In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

# Step 1: list endpoints
endpoints = client.list_endpoints()
print(endpoints)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'endpoints': [{'name': 'rbi_circular_vs_endpoint', 'creator': 'ee240002079@iiti.ac.in', 'creation_timestamp': 1776493719617, 'last_updated_timestamp': 1776493719617, 'endpoint_type': 'STANDARD', 'last_updated_user': 'ee240002079@iiti.ac.in', 'id': 'e1c69b4f-50d5-4369-87b5-51c4bb031612', 'endpoint_status': {'state': 'ONLINE'}, 'num_indexes': 1}]}


In [0]:
from databricks.vector_search.client import VectorSearchClient

endpoint_name = "rbi_circular_vs_endpoint"
index_name = "workspace.default.gold_rbi_circular_chunks_index"

client = VectorSearchClient()

print(client.list_indexes(endpoint_name))

index = client.get_index(endpoint_name=endpoint_name, index_name=index_name)
print(index.describe())

try:
    index.wait_until_ready(wait_for_updates=True, verbose=True)
    print("Index is ready")
except Exception as e:
    print("Index is still not ready")
    print(str(e))


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'vector_indexes': [{'name': 'workspace.default.gold_rbi_circular_chunks_index', 'endpoint_name': 'rbi_circular_vs_endpoint', 'primary_key': 'chunk_id', 'index_type': 'DELTA_SYNC', 'creator': 'ee240002079@iiti.ac.in', 'id': '6d60052c-3d8a-4ad6-b7f1-80f4224ffa01'}]}
{'name': 'workspace.default.gold_rbi_circular_chunks_index', 'endpoint_name': 'rbi_circular_vs_endpoint', 'primary_key': 'chunk_id', 'index_type': 'DELTA_SYNC', 'delta_sync_index_spec': {'source_table': 'workspace.default.gold_rbi_circular_chunks', 'embedding_source_columns': [{'name': 'retrieval_text', 'embedding_model_endpoint_name': 'databricks-gte-large-en'}], 'pipeline_type': 'TRIGGERED', 'pipeline_id': 'cb49ca81-cd28-4518-b627-db9dc97904d9'}, 'status': {'detailed_state': 'PROVISIONING_ENDPOINT', 'message': 'Delta

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
index = client.get_index(
    endpoint_name="rbi_circular_vs_endpoint",
    index_name="workspace.default.gold_rbi_circular_chunks_index"
)

results = index.similarity_search(
    query_text="What did RBI say about KYC updates?",
    columns=["chunk_id", "document", "chunks_text", "issued_on", "regulation_area", "applicable_to"],
    num_results=3
)

print(results)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'manifest': {'column_count': 7, 'columns': [{'name': 'chunk_id'}, {'name': 'document'}, {'name': 'chunks_text'}, {'name': 'issued_on'}, {'name': 'regulation_area'}, {'name': 'applicable_to'}, {'name': 'score'}]}, 'result': {'row_count': 3, 'data_array': [['5de2639c055ea0188762fece2376133d466c23a485860a44f7d29f7751b30f9d', 'RBI_2024-2025_87DOR.AML.REC.49_14.01.001_2024-25_2024-11-06', '![](_page_0_Picture_0.jpeg) RBI/2024-2025/87 DOR.AML.REC.49/14.01.001/2024-25 November 06, 2024 All the Regulated Entities Dear Sir/Madam, **Amendment to the Master Direction - Know Your Cust

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
index = client.get_index(
    endpoint_name="rbi_circular_vs_endpoint",
    index_name="workspace.default.gold_rbi_circular_chunks_index"
)

def retrieve_rbi_context(question, k=3):
    res = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "document", "chunks_text", "issued_on", "regulation_area", "applicable_to"],
        num_results=k
    )
    rows = res["result"]["data_array"]

    contexts = []
    for r in rows:
        contexts.append({
            "chunk_id": r[0],
            "document": r[1],
            "chunks_text": r[2],
            "issued_on": r[3],
            "regulation_area": r[4],
            "applicable_to": r[5],
            "score": r[6],
        })
    return contexts

def build_context_text(contexts):
    return "\n\n".join([
        f"""[Chunk ID: {c['chunk_id']}]
Document: {c['document']}
Issued on: {c['issued_on']}
Regulation area: {c['regulation_area']}
Applicable to: {c['applicable_to']}
Text: {c['chunks_text']}"""
        for c in contexts
    ])

question = "What did RBI say about KYC updates?"
contexts = retrieve_rbi_context(question, k=3)
context_text = build_context_text(contexts)

print(context_text[:5000])


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[Chunk ID: 5de2639c055ea0188762fece2376133d466c23a485860a44f7d29f7751b30f9d]
Document: RBI_2024-2025_87DOR.AML.REC.49_14.01.001_2024-25_2024-11-06
Issued on: 2024-11-06
Regulation area: Anti-Money Laundering
Applicable to: All the Regulated Entities
Text: ![](_page_0_Picture_0.jpeg) RBI/2024-2025/87 DOR.AML.REC.49/14.01.001/2024-25 November 06, 2024 All the Regulated Entities Dear Sir/Madam, **Amendment to the Master Direction - Know Your Customer (KYC) Direction, 2016** Please refer to the Master Direction [- Know Your Customer](https://www.rbi.org.in/Scripts/BS_ViewMasDir

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
index = client.get_index(
    endpoint_name="rbi_circular_vs_endpoint",
    index_name="workspace.default.gold_rbi_circular_chunks_index"
)

def retrieve_rbi_context(question, k=3):
    res = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "document", "chunks_text", "issued_on", "regulation_area", "applicable_to"],
        num_results=k
    )
    rows = res["result"]["data_array"]

    contexts = []
    for r in rows:
        contexts.append({
            "chunk_id": r[0],
            "document": r[1],
            "chunks_text": r[2],
            "issued_on": r[3],
            "regulation_area": r[4],
            "applicable_to": r[5],
            "score": r[6],
        })
    return contexts

def build_context_text(contexts):
    return "\n\n".join([
        f"""[Chunk ID: {c['chunk_id']}]
Document: {c['document']}
Issued on: {c['issued_on']}
Regulation area: {c['regulation_area']}
Applicable to: {c['applicable_to']}
Text: {c['chunks_text']}"""
        for c in contexts
    ])

def build_prompt(question, context_text):
    return f"""
You are an RBI circular explainer for ordinary Indian users.

Answer only from the provided RBI context.
If the answer is not clearly present in the context, say that clearly.

Question:
{question}

RBI Context:
{context_text}

Return the answer in this format:

1. Simple explanation
2. Who it applies to
3. Key action points
4. Source chunk IDs used
"""


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
question = "What did RBI say about KYC updates?"
contexts = retrieve_rbi_context(question, k=3)
context_text = build_context_text(contexts)
prompt = build_prompt(question, context_text)

print(prompt[:6000])


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

You are an RBI circular explainer for ordinary Indian users.

Answer only from the provided RBI context.
If the answer is not clearly present in the context, say that clearly.

Question:
What did RBI say about KYC updates?

RBI Context:
[Chunk ID: 5de2639c055ea0188762fece2376133d466c23a485860a44f7d29f7751b30f9d]
Document: RBI_2024-2025_87DOR.AML.REC.49_14.01.001_2024-25_2024-11-06
Issued on: 2024-11-06
Regulation area: Anti-Money Laundering
Applicable to: All the Regulated Entities
Text: ![](_page_0_Picture_0.jpeg) RBI/2024-2025/87 DOR.AML.REC.49/14.01.001/2024-25 November 06, 2024 All the Regulated Entities Dear Sir/Madam, **Amendment to the Master Direction - Know Your Customer (KYC) Direction, 2016** Please refer to the Master Direction [- Know Your Customer](https://www.rbi.

In [0]:
%pip install -U mlflow transformers torch accelerate sentencepiece


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:
%pip install -U openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-294e45e7-8d19-4b8f-a9ce-abc6026f1824
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: openai
    Found existing installation: openai 2.14.0
    Not uninstalling openai at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-294e45e7-8d19-4b8f-a9ce-abc6026f1824
    Can't uninstall 'openai'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:
import os
from openai import OpenAI

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints"
)

response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain in 3 lines what RBI KYC updates generally mean."}
    ],
    max_tokens=200
)

print(response.choices[0].message.content)


RBI KYC (Know Your Customer) updates refer to the periodic verification of a customer's identity and address by banks and financial institutions. This process helps prevent money laundering and ensures that accounts are not used for illicit activities. RBI mandates KYC updates to maintain the integrity of the financial system and protect customer interests.


In [0]:
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI

vs_client = VectorSearchClient()
index = vs_client.get_index(
    endpoint_name="rbi_circular_vs_endpoint",
    index_name="workspace.default.gold_rbi_circular_chunks_index"
)

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

llm_client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints"
)

def retrieve_rbi_context(question, k=3):
    res = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "document", "chunks_text", "issued_on", "regulation_area", "applicable_to"],
        num_results=k
    )
    rows = res["result"]["data_array"]
    return [
        {
            "chunk_id": r[0],
            "document": r[1],
            "chunks_text": r[2],
            "issued_on": r[3],
            "regulation_area": r[4],
            "applicable_to": r[5],
            "score": r[6],
        }
        for r in rows
    ]

def build_context_text(contexts):
    return "\n\n".join([
        f"""[Chunk ID: {c['chunk_id']}]
Document: {c['document']}
Issued on: {c['issued_on']}
Regulation area: {c['regulation_area']}
Applicable to: {c['applicable_to']}
Text: {c['chunks_text']}"""
        for c in contexts
    ])

question = "What did RBI say about KYC updates?"
contexts = retrieve_rbi_context(question, k=3)
context_text = build_context_text(contexts)

messages = [
    {
        "role": "system",
        "content": "You are an RBI circular explainer for ordinary Indian users. Answer only from the provided RBI context. If the answer is not present in the context, say so clearly. Write in simple Hindi."
    },
    {
        "role": "user",
        "content": f"""Question:
{question}

RBI Context:
{context_text}

Return the answer in this format:
1. Simple explanation
2. Who it applies to
3. Key action points
4. Source chunk IDs used"""
    }
]

response = llm_client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=messages,
    max_tokens=600
)

print(response.choices[0].message.content)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
1. सरल व्याख्या:
आरबीआई ने ग्राहक की जानकारी को अद्यतन करने के लिए निर्देश जारी किए हैं। इसमें कहा गया है कि यदि एक मौजूदा ग्राहक एक ही संस्था से दूसरा खाता खोलना चाहता है, तो उसे फिर से ग्राहक की जानकारी देने की आवश्यकता नहीं है। इसके अलावा, संस्थाओं को अपने ग्राहकों की जानकारी को नियमित रूप से अद्यतन करना होगा और इस जानकारी को सेंट्रल केवाईसी रिकॉर्ड्स रजिस्ट्री में अपलोड करना होगा।
2. यह किस पर लागू होता है:
यह निर्देश सभी विनियमित संस्थाओं पर लागू होता है।
3. महत्वपूर्ण कार्रवाई बिंदु:
- यदि एक मौजूदा ग्राहक एक ही संस्था से दूसरा खाता खोलना चाहता है, तो उसे फिर से ग्राह

In [0]:
def translate_hindi_to_tamil_llama(hindi_text):
    """Use Llama for Hindi to Tamil translation"""
    messages = [
        {
            "role": "system",
            "content": "You are a professional Hindi to Tamil translator. Translate the given Hindi text to Tamil accurately, preserving the meaning and tone."
        },
        {
            "role": "user",
            "content": f"Translate the following Hindi text to Tamil:\n\n{hindi_text}"
        }
    ]
    
    response = llm_client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=messages,
        max_tokens=800,
        temperature=0.3  # Lower temperature for more consistent translation
    )
    
    return response.choices[0].message.content

In [0]:
# Complete 3-Step Pipeline Test

question = "What did RBI say about KYC updates?"

print("\n" + "="*80)
print("STEP 1: Retrieving relevant RBI circular chunks...")
print("="*80)
contexts = retrieve_rbi_context(question, k=3)
context_text = build_context_text(contexts)
print(f"Retrieved {len(contexts)} chunks")
for ctx in contexts:
    print(f"  - {ctx['document'][:50]}... (score: {ctx['score']:.4f})")

print("\n" + "="*80)
print("STEP 2: Generating answer in Hindi using Databricks Llama...")
print("="*80)
messages = [
    {
        "role": "system",
        "content": "You are an RBI circular explainer for ordinary Indian users. Answer only from the provided RBI context. If the answer is not present in the context, say so clearly. Write in simple Hindi (Devanagari script)."
    },
    {
        "role": "user",
        "content": f"""Question:
{question}

RBI Context:
{context_text}

Return the answer in Hindi in this format:
1. सरल विवरण (Simple explanation)
2. यह किन पर लागू होता है (Who it applies to)
3. मुख्य कार्रवाई बिन्दु (Key action points)
4. स्रोत चंक आईडी उपयोग किए गए (Source chunk IDs used)"""
    }
]

hindi_response = llm_client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=messages,
    max_tokens=800
)

hindi_answer = hindi_response.choices[0].message.content
print("\nHINDI ANSWER:")
print(hindi_answer)

print("\n" + "="*80)
print("STEP 3: Translating Hindi to Tamil using Databricks Llama...")
print("="*80)
tamil_answer = translate_hindi_to_tamil_llama(hindi_answer)
print("\nTAMIL ANSWER:")
print(tamil_answer)

print("\n" + "="*80)
print("✅ COMPLETE PIPELINE EXECUTED SUCCESSFULLY")
print("="*80)


STEP 1: Retrieving relevant RBI circular chunks...
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Retrieved 3 chunks
  - RBI_2024-2025_87DOR.AML.REC.49_14.01.001_2024-25_2... (score: 0.6871)
  - RBI_2024-2025_87DOR.AML.REC.49_14.01.001_2024-25_2... (score: 0.6522)
  - RBI_2023-2024_24DOR.AML.REC.111_14.01.001_2023-24_... (score: 0.6499)

STEP 2: Generating answer in Hindi using Databricks Llama...

HINDI ANSWER:
1. सरल विवरण: भारतीय रिज़र्व बैंक (आरबीआई) ने ग्राहक को जानने के निर्देश (केवाईसी) में संशोधन किया है। इसमें ग्राहक की जानकारी को अद्यतन करने और केंद्रीय केवाईसी रिकॉर्ड रजिस्ट्री (सीकेवाईसीआर) में जानकारी अपलोड करने के बारे में निर्देश दिए गए हैं।
2. यह किन पर लागू होता है: यह निर्देश सभी नियंत्रित संस्थाओं (रेगुलेटेड एंटिटीज) पर लागू होता है।
3. मुख्य कार्रवाई बिन्दु: 
   - ग्राहक की जानकारी को अद्यतन करने के लिए न

In [0]:
def answer_rbi_question(question: str, target_language: str = "tamil", num_chunks: int = 3) -> dict:
    """
    Complete RAG pipeline for RBI circular questions with multilingual support.
    
    Args:
        question: User's question about RBI circulars
        target_language: Output language ("hindi", "tamil", "english", "telugu", "kannada", "malayalam")
        num_chunks: Number of context chunks to retrieve (default: 3)
    
    Returns:
        dict with keys: question, contexts, hindi_answer, translated_answer, target_language
    """
    # Step 1: Retrieve relevant RBI context
    contexts = retrieve_rbi_context(question, k=num_chunks)
    context_text = build_context_text(contexts)
    
    # Step 2: Generate Hindi answer
    messages = [
        {
            "role": "system",
            "content": "You are an RBI circular explainer for ordinary Indian users. Answer only from the provided RBI context. If the answer is not present in the context, say so clearly. Write in simple Hindi (Devanagari script)."
        },
        {
            "role": "user",
            "content": f"""Question:
{question}

RBI Context:
{context_text}

Return the answer in Hindi in this format:
1. सरल विवरण (Simple explanation)
2. यह किन पर लागू होता है (Who it applies to)
3. मुख्य कार्रवाई बिन्दु (Key action points)
4. स्रोत चंक आईडी उपयोग किए गए (Source chunk IDs used)"""
        }
    ]
    
    hindi_response = llm_client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=messages,
        max_tokens=800
    )
    hindi_answer = hindi_response.choices[0].message.content
    
    # Step 3: Translate to target language (if not Hindi)
    if target_language.lower() == "hindi":
        translated_answer = hindi_answer
    else:
        # Language mapping
        lang_map = {
            "tamil": "Tamil",
            "english": "English",
            "telugu": "Telugu",
            "kannada": "Kannada",
            "malayalam": "Malayalam",
            "bengali": "Bengali",
            "marathi": "Marathi",
            "gujarati": "Gujarati"
        }
        
        target_lang_name = lang_map.get(target_language.lower(), "English")
        
        translation_messages = [
            {
                "role": "system",
                "content": f"You are a professional Hindi to {target_lang_name} translator. Translate the given Hindi text to {target_lang_name} accurately, preserving the meaning and tone."
            },
            {
                "role": "user",
                "content": f"Translate the following Hindi text to {target_lang_name}:\n\n{hindi_answer}"
            }
        ]
        
        translation_response = llm_client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=translation_messages,
            max_tokens=800,
            temperature=0.3
        )
        translated_answer = translation_response.choices[0].message.content
    
    return {
        "question": question,
        "contexts": contexts,
        "hindi_answer": hindi_answer,
        "translated_answer": translated_answer,
        "target_language": target_language,
        "num_chunks_retrieved": len(contexts)
    }

# Test the function
print("Testing reusable RAG function...\n")
result = answer_rbi_question(
    question="What are the new rules for digital payments?",
    target_language="tamil",
    num_chunks=3
)

print(f"Question: {result['question']}")
print(f"\nRetrieved {result['num_chunks_retrieved']} chunks")
print(f"\nHindi Answer:\n{result['hindi_answer']}")
print(f"\n{result['target_language'].upper()} Answer:\n{result['translated_answer']}")
print("\n✅ Reusable function works!")

Testing reusable RAG function...

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Question: What are the new rules for digital payments?

Retrieved 3 chunks

Hindi Answer:
1. सरल विवरण: नए नियमों के अनुसार, डिजिटल भुगतान प्रणाली को व्यक्तियों के साथ विकलांगता के लिए अधिक सुलभ बनाने के लिए डिज़ाइन किया जाना चाहिए। इसके अलावा, छोटे मूल्य के डिजिटल भुगतान ऑफ़लाइन मोड में किए जा सकते हैं, जिसमें ₹500 तक की सीमा होगी और कुल सीमा ₹2,000 होगी।

2. यह किन पर लागू होता है: यह नियम सभी भुगतान प्रणाली प्रतिभागियों (पीएसपी), जैसे कि बैंक और अधिकृत गैर-बैंक भुगतान प्रणाली प्रदाताओं पर लागू होता है।

3. मुख्य कार्रवाई बिन्दु: 
- भुगतान प्रणाली प्रतिभागियों को अपनी प्रणालियों और उपकरणों की समीक्षा करनी चाहिए और विकलांग व्यक्तियों के लिए सुलभ बनाने के लिए आवश्यक संशोधन करने चाहिए।
- ऑफ़लाइन भुगतान लेनदेन के लिए, ₹500 तक की सीमा होगी और कुल सी

In [0]:
# Create Databricks App (app.py content)
# Save this as app.py in your workspace

app_code = '''
import streamlit as st
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI
import os

# Initialize clients
vs_client = VectorSearchClient()
index = vs_client.get_index(
    endpoint_name="rbi_circular_vs_endpoint",
    index_name="workspace.default.gold_rbi_circular_chunks_index"
)

host = os.environ.get("DATABRICKS_HOST")
token = os.environ.get("DATABRICKS_TOKEN")

llm_client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints"
)

# Helper functions
def retrieve_rbi_context(question, k=3):
    res = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "document", "chunks_text", "issued_on", "regulation_area", "applicable_to"],
        num_results=k
    )
    rows = res["result"]["data_array"]
    return [
        {
            "chunk_id": r[0],
            "document": r[1],
            "chunks_text": r[2],
            "issued_on": r[3],
            "regulation_area": r[4],
            "applicable_to": r[5],
            "score": r[6],
        }
        for r in rows
    ]

def build_context_text(contexts):
    return "\\n\\n".join([
        f"""[Chunk ID: {c[\'chunk_id\']}]
Document: {c[\'document\']}
Issued on: {c[\'issued_on\']}
Regulation area: {c[\'regulation_area\']}
Applicable to: {c[\'applicable_to\']}
Text: {c[\'chunks_text\']}"""  
        for c in contexts
    ])

def generate_and_translate(question, target_language, num_chunks):
    # Retrieve context
    contexts = retrieve_rbi_context(question, k=num_chunks)
    context_text = build_context_text(contexts)
    
    # Generate Hindi answer
    messages = [
        {
            "role": "system",
            "content": "You are an RBI circular explainer for ordinary Indian users. Answer only from the provided RBI context. Write in simple Hindi."
        },
        {
            "role": "user",
            "content": f"""Question: {question}\n\nRBI Context:\n{context_text}\n\nReturn the answer in Hindi."""
        }
    ]
    
    hindi_response = llm_client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=messages,
        max_tokens=800
    )
    hindi_answer = hindi_response.choices[0].message.content
    
    # Translate if needed
    if target_language != "Hindi":
        translation_messages = [
            {"role": "system", "content": f"Translate Hindi to {target_language}."},
            {"role": "user", "content": f"Translate:\\n\\n{hindi_answer}"}
        ]
        translation_response = llm_client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=translation_messages,
            max_tokens=800,
            temperature=0.3
        )
        translated_answer = translation_response.choices[0].message.content
    else:
        translated_answer = hindi_answer
    
    return hindi_answer, translated_answer, contexts

# Streamlit UI
st.title("🏦 RBI Circular Assistant")
st.markdown("Ask questions about RBI circulars in your preferred language")

# Sidebar settings
with st.sidebar:
    st.header("Settings")
    target_language = st.selectbox(
        "Output Language",
        ["Hindi", "Tamil", "English", "Telugu", "Kannada", "Malayalam"]
    )
    num_chunks = st.slider("Number of context chunks", 1, 5, 3)
    st.markdown("---")
    st.markdown("### About")
    st.markdown("This app uses RAG to answer questions about RBI circulars using Databricks Foundation Models.")

# Main interface
question = st.text_input("Your Question:", placeholder="e.g., What are the new KYC requirements?")

if st.button("Get Answer", type="primary"):
    if question:
        with st.spinner("Searching RBI circulars and generating answer..."):
            hindi_answer, translated_answer, contexts = generate_and_translate(
                question, target_language, num_chunks
            )
        
        # Display results
        st.success("Answer generated!")
        
        col1, col2 = st.columns(2)
        
        with col1:
            st.subheader("📝 Hindi Answer")
            st.write(hindi_answer)
        
        with col2:
            st.subheader(f"📝 {target_language} Answer")
            st.write(translated_answer)
        
        # Show source documents
        with st.expander("📚 Source Documents"):
            for i, ctx in enumerate(contexts):
                st.markdown(f"**{i+1}. {ctx[\'document\']}** (Score: {ctx[\'score\']:.4f})")
                st.text(f"Issued: {ctx[\'issued_on\']} | Area: {ctx[\'regulation_area\']}")
                st.caption(ctx[\'chunks_text\'][:300] + "...")
                st.markdown("---")
    else:
        st.warning("Please enter a question")
'''

print("Databricks App Code Generated!")
print("\nTo deploy:")
print("1. Create a file 'app.py' in your workspace")
print("2. Copy the code above into app.py")
print("3. Use Databricks Apps to deploy it")
print("\nSaving app code to file...")

# Save to workspace
with open("/Workspace/Users/ee240002079@iiti.ac.in/rbi_app.py", "w") as f:
    f.write(app_code)

print("\n✅ App code saved to: /Workspace/Users/ee240002079@iiti.ac.in/rbi_app.py")

Databricks App Code Generated!

To deploy:
1. Create a file 'app.py' in your workspace
2. Copy the code above into app.py
3. Use Databricks Apps to deploy it

Saving app code to file...

✅ App code saved to: /Workspace/Users/ee240002079@iiti.ac.in/rbi_app.py


In [0]:
import requests
import time
from datetime import datetime

def check_endpoint_status(endpoint_name="rbi-rag-endpoint", poll_interval=30, max_wait_minutes=15):
    """
    Monitor Model Serving endpoint deployment status
    """
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = ctx.apiUrl().get()
    token = ctx.apiToken().get()
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    print(f"🔍 Monitoring endpoint: {endpoint_name}")
    print(f"⏰ Poll interval: {poll_interval} seconds")
    print(f"⏱️  Max wait time: {max_wait_minutes} minutes")
    print("\n" + "="*80 + "\n")
    
    start_time = time.time()
    max_wait_seconds = max_wait_minutes * 60
    
    while (time.time() - start_time) < max_wait_seconds:
        try:
            response = requests.get(
                f"{host}/api/2.0/serving-endpoints/{endpoint_name}",
                headers=headers
            )
            
            if response.status_code == 200:
                data = response.json()
                state = data.get("state", {})
                
                ready = state.get("ready", "UNKNOWN")
                config_update = state.get("config_update", "UNKNOWN")
                
                # Get current/pending model versions
                config = data.get("config", {})
                pending_config = data.get("pending_config", {})
                
                current_version = "N/A"
                pending_version = "N/A"
                
                if config.get("served_models"):
                    current_version = config["served_models"][0].get("model_version", "N/A")
                
                if pending_config.get("served_models"):
                    pending_version = pending_config["served_models"][0].get("model_version", "N/A")
                
                timestamp = datetime.now().strftime("%H:%M:%S")
                
                print(f"[{timestamp}] Status:")
                print(f"  Ready: {ready}")
                print(f"  Config Update: {config_update}")
                print(f"  Current Version: {current_version}")
                
                if pending_version != "N/A":
                    print(f"  🔄 Pending Version: {pending_version}")
                
                # Success conditions
                if ready == "READY" and config_update == "NOT_UPDATING":
                    print("\n" + "="*80)
                    print("✅ DEPLOYMENT SUCCESSFUL!")
                    print("="*80)
                    print(f"\n🎉 Endpoint is ready with Version {current_version}")
                    print(f"\n🔗 Test your endpoint at:")
                    print(f"   {host}/serving-endpoints/{endpoint_name}/invocations")
                    return True
                
                # Failure condition
                if config_update == "UPDATE_FAILED":
                    print("\n" + "="*80)
                    print("❌ DEPLOYMENT FAILED")
                    print("="*80)
                    print("\n📋 Check the endpoint logs for details.")
                    return False
                
                print("  ⏳ Waiting...\n")
            else:
                print(f"[{timestamp}] ❌ Error: {response.status_code}")
        
        except Exception as e:
            print(f"[{timestamp}] ⚠️  Exception: {str(e)}")
        
        time.sleep(poll_interval)
    
    print("\n" + "="*80)
    print("⏰ TIMEOUT - Deployment taking longer than expected")
    print("="*80)
    print("\nCheck the endpoint manually in the Databricks UI.")
    return False

# Run the monitoring
print("🚀 Starting endpoint deployment monitoring...\n")
print("💡 TIP: While this runs, complete Step 3 in the UI to update to Version 4\n")

check_endpoint_status()

In [0]:
# OPTIONAL: Create a fresh endpoint if updating fails
# Only run this if you can't update the existing endpoint

import requests
import json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

endpoint_config = {
    "name": "rbi-rag-endpoint-v4",  # New endpoint name
    "config": {
        "served_models": [
            {
                "model_name": "workspace.default.rbi_rag_model",
                "model_version": "4",
                "workload_size": "Small",
                "scale_to_zero_enabled": True
            }
        ]
    }
}

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

response = requests.post(
    f"{host}/api/2.0/serving-endpoints",
    headers=headers,
    json=endpoint_config
)

if response.status_code in [200, 201]:
    print("✅ New endpoint created successfully!")
    print(f"\nEndpoint name: rbi-rag-endpoint-v4")
    print(f"\nMonitor deployment at:")
    print(f"{host}/ml/endpoints/rbi-rag-endpoint-v4")
else:
    print(f"❌ Error creating endpoint: {response.status_code}")
    print(response.text)

In [0]:
import mlflow
import pandas as pd
import requests
import json
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Define custom PyFunc model for RAG pipeline
class RBIRAGModel(mlflow.pyfunc.PythonModel):
    def __init__(self, endpoint_name, index_name, vector_endpoint):
        self.endpoint_name = endpoint_name
        self.index_name = index_name
        self.vector_endpoint = vector_endpoint
    
    def load_context(self, context):
        """Load clients when model is loaded"""
        from openai import OpenAI
        import os
        
        # Get credentials from serving environment
        self.workspace_url = os.environ.get("DATABRICKS_HOST")
        self.token = os.environ.get("DATABRICKS_TOKEN")
        
        # Store index details (will use REST API instead of SDK)
        self.index_endpoint = self.vector_endpoint
        self.index_name_full = self.index_name
        
        # Initialize LLM Client
        self.llm_client = OpenAI(
            api_key=self.token,
            base_url=f"{self.workspace_url}/serving-endpoints"
        )
    
    def _query_vector_index(self, query_text, num_results=3):
        """Query vector index using REST API instead of SDK"""
        url = f"{self.workspace_url}/api/2.0/vector-search/indexes/{self.index_name_full}/query"
        
        headers = {
            "Authorization": f"Bearer {self.token}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "query_text": query_text,
            "columns": ["chunk_id", "document", "chunks_text", "issued_on", "regulation_area", "applicable_to"],
            "num_results": num_results
        }
        
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        
        result = response.json()
        rows = result.get("result", {}).get("data_array", [])
        
        return [{
            "document": r[1],
            "chunks_text": r[2],
            "issued_on": r[3],
            "regulation_area": r[4],
            "applicable_to": r[5],
        } for r in rows]
    
    def predict(self, context, model_input):
        """Generate answers for input questions"""
        results = []
        
        for _, row in model_input.iterrows():
            question = row["question"]
            target_language = row.get("target_language", "tamil")
            num_chunks = int(row.get("num_chunks", 3))
            
            # Retrieve context using REST API
            contexts = self._query_vector_index(question, num_chunks)
            
            context_text = "\n\n".join([
                f"""Document: {c['document']}
Issued on: {c['issued_on']}
Text: {c['chunks_text']}"""
                for c in contexts
            ])
            
            # Generate Hindi answer
            messages = [
                {"role": "system", "content": "You are an RBI circular explainer. Answer in Hindi."},
                {"role": "user", "content": f"Question: {question}\n\nContext:\n{context_text}"}
            ]
            
            hindi_response = self.llm_client.chat.completions.create(
                model="databricks-meta-llama-3-3-70b-instruct",
                messages=messages,
                max_tokens=800
            )
            hindi_answer = hindi_response.choices[0].message.content
            
            # Translate if needed
            if target_language.lower() != "hindi":
                translation_messages = [
                    {"role": "system", "content": f"Translate Hindi to {target_language}."},
                    {"role": "user", "content": f"Translate:\n\n{hindi_answer}"}
                ]
                translation_response = self.llm_client.chat.completions.create(
                    model="databricks-meta-llama-3-3-70b-instruct",
                    messages=translation_messages,
                    max_tokens=800,
                    temperature=0.3
                )
                final_answer = translation_response.choices[0].message.content
            else:
                final_answer = hindi_answer
            
            results.append({
                "question": question,
                "answer": final_answer,
                "language": target_language
            })
        
        return pd.DataFrame(results)

# Define model signature
input_schema = Schema([
    ColSpec("string", "question"),
    ColSpec("string", "target_language"),
    ColSpec("long", "num_chunks")
])

output_schema = Schema([
    ColSpec("string", "question"),
    ColSpec("string", "answer"),
    ColSpec("string", "language")
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Create sample input
sample_input = pd.DataFrame({
    "question": ["What are the KYC requirements?"],
    "target_language": ["tamil"],
    "num_chunks": [3]
})

# Log model to MLflow
mlflow.set_registry_uri("databricks-uc")

catalog = "workspace"
schema = "default"
model_name = "rbi_rag_model"
model_fqn = f"{catalog}.{schema}.{model_name}"

with mlflow.start_run(run_name="rbi_rag_pipeline_v5_rest_api") as run:
    # Create model instance
    model = RBIRAGModel(
        endpoint_name="databricks-meta-llama-3-3-70b-instruct",
        index_name="workspace.default.gold_rbi_circular_chunks_index",
        vector_endpoint="rbi_circular_vs_endpoint"
    )
    
    # Log model
    model_info = mlflow.pyfunc.log_model(
        artifact_path="rbi_rag_model",
        python_model=model,
        signature=signature,
        input_example=sample_input,
        pip_requirements=[
            "openai",
            "pandas",
            "requests"  # Added for REST API
        ],
        registered_model_name=model_fqn
    )
    
    print(f"✅ Model logged: {model_info.model_uri}")
    print(f"✅ Model registered: {model_fqn}")
    print(f"\nRun ID: {run.info.run_id}")
    print(f"\n🔧 USES REST API FOR VECTOR SEARCH (not SDK)")
    print(f"\nTo deploy:")
    print(f"1. Go to Serving -> Model Serving -> rbi-rag-endpoint")
    print(f"2. Edit configuration")
    print(f"3. Change to Version 5")
    print(f"4. Save and wait for deployment")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-71767dd7-9712-49ef-9779-8de7682859ce/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/04/18 10:06:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-97058e79-1ee1.cloud.databricks.com/ml/experiments/4385286339124170/models/m-abd91148be974f9cbe1d8aba519bad20?o=7474652524255061
2026/04/18 10:06:22 INFO mlflow.pyfunc: Validating input example against model signature
2026/04/18 10:06:22 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "q.... Alternatively, you can avoid passing input example and pass model signature inste

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '5' of model 'workspace.default.rbi_rag_model': https://dbc-97058e79-1ee1.cloud.databricks.com/explore/data/models/workspace/default/rbi_rag_model/version/5?o=7474652524255061


✅ Model logged: models:/m-abd91148be974f9cbe1d8aba519bad20
✅ Model registered: workspace.default.rbi_rag_model

Run ID: 8554a3b3cea2420f8b68287bd6dc2d50

🔧 USES REST API FOR VECTOR SEARCH (not SDK)

To deploy:
1. Go to Serving -> Model Serving -> rbi-rag-endpoint
2. Edit configuration
3. Change to Version 5
4. Save and wait for deployment


In [0]:
from pyspark.sql import functions as F
import pandas as pd

# Load evaluation dataset
eval_table = "workspace.default.gold_rbi_eval"
eval_df = spark.table(eval_table)

print(f"Evaluation dataset: {eval_df.count()} questions")
print("\nSample evaluation questions:")
display(eval_df.select("question", "answer", "category", "estimated_difficulty").limit(5))

# Evaluation function
def evaluate_rag_answer(question, expected_answer, generated_answer, category):
    """
    Evaluate RAG answer quality using LLM as judge
    """
    evaluation_prompt = f"""You are an expert evaluator for a question-answering system about RBI circulars.

Evaluate the generated answer against the expected answer based on:
1. Accuracy (0-10): Does it contain correct information from the context?
2. Completeness (0-10): Does it cover all key points from the expected answer?
3. Clarity (0-10): Is it clear and easy to understand?
4. Relevance (0-10): Does it directly answer the question?

Question: {question}

Expected Answer:
{expected_answer}

Generated Answer:
{generated_answer}

Return ONLY a JSON with scores and brief reasoning:
{{
  "accuracy": <score>,
  "completeness": <score>,
  "clarity": <score>,
  "relevance": <score>,
  "overall_score": <average>,
  "reasoning": "<brief explanation>"
}}"""
    
    messages = [
        {"role": "system", "content": "You are an evaluation expert. Return only valid JSON."},
        {"role": "user", "content": evaluation_prompt}
    ]
    
    response = llm_client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=messages,
        max_tokens=500,
        temperature=0.1
    )
    
    return response.choices[0].message.content

# Run evaluation on sample questions
print("\n" + "="*80)
print("Running Evaluation on Sample Questions")
print("="*80)

eval_sample = eval_df.limit(3).toPandas()
evaluation_results = []

for idx, row in eval_sample.iterrows():
    print(f"\n[{idx+1}/{len(eval_sample)}] Evaluating: {row['question'][:60]}...")
    
    # Generate answer using RAG pipeline
    result = answer_rbi_question(
        question=row['question'],
        target_language="hindi",
        num_chunks=3
    )
    
    generated_answer = result['hindi_answer']
    
    # Evaluate
    eval_result = evaluate_rag_answer(
        question=row['question'],
        expected_answer=row['answer'],
        generated_answer=generated_answer,
        category=row['category']
    )
    
    evaluation_results.append({
        'qa_id': row['qa_id'],
        'question': row['question'],
        'category': row['category'],
        'difficulty': row['estimated_difficulty'],
        'expected_answer': row['answer'],
        'generated_answer': generated_answer,
        'evaluation': eval_result
    })
    
    print(f"Evaluation: {eval_result[:200]}...")

# Save evaluation results
eval_results_df = spark.createDataFrame(pd.DataFrame(evaluation_results))
eval_results_table = "workspace.default.rbi_rag_evaluation_results"

eval_results_df.write.format("delta").mode("overwrite").saveAsTable(eval_results_table)

print(f"\n✅ Evaluation complete! Results saved to: {eval_results_table}")
print(f"\nTo view results: spark.table('{eval_results_table}')")

# Display summary
print("\n" + "="*80)
print("Evaluation Summary")
print("="*80)
display(eval_results_df)

Evaluation dataset: 1000 questions

Sample evaluation questions:


question,answer,category,estimated_difficulty
What forms of assets can be used for posting and collecting margin in India by Authorised Dealer Category-I banks for permitted derivative contracts?,"Authorised Dealer Category-I (AD Cat-I) banks are permitted to post and collect margin in India for permitted derivative contracts using Indian currency, freely convertible foreign currency, debt securities issued by the Indian Central Government and State Governments, and Rupee bonds issued by persons resident in India that are listed on a recognized stock exchange in India and have a credit rating of AAA issued by a rating agency registered with the Securities and Exchange Board of India.",fact-based,5
"What are the requirements for banks to reconcile their SGL account balances with the Public Debt Office (PDO), and what reporting is required?","Banks are required to reconcile their SGL account balances with the balances in the books of the Public Debt Offices (PDOs) monthly. The reconciliation results should be presented to the Audit Committee of the Board, and the internal audit department should periodically check this reconciliation. Additionally, banks must send a quarterly certificate to the PDO confirming the SGL account balances have been reconciled and presented before the appropriate committee.",fact-based,5
How do Real Time Gross Settlement (RTGS) and National Electronic Funds Transfer (NEFT) relate to India's electronic funds transfer infrastructure?,The Electronic Funds Transfer (EFT) infrastructure in India utilizes both Real Time Gross Settlement (RTGS) and National Electronic Funds Transfer (NEFT) systems to facilitate the transfer of funds electronically between individuals and entities.,fact-based,3
What role do banks play in promoting employment and providing credit facilities to marginalized communities in India?,"Banks play a role in promoting employment, particularly regarding agricultural credit to small and marginal farmers, and by providing credit facilities to Scheduled Castes and Scheduled Tribes.",fact-based,5
What is the asset classification of project loans when the Date of Commencement of Commercial Operations (DCCO) is extended and cost overruns are funded according to specified conditions?,"When a project's Date of Commencement of Commercial Operations (DCCO) is extended and cost overruns are funded in compliance with specified thresholds, the loans are classified as 'standard' assets.",fact-based,2



Running Evaluation on Sample Questions

[1/3] Evaluating: What forms of assets can be used for posting and collecting ...
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Evaluation: ```
{
  "accuracy": 9,
  "completeness": 8,
  "clarity": 7,
  "relevance": 9,
  "overall_score": 8.25,
  "reasoning": "The generated answer accurately covers the key points from the expected answer, i...

[2/3] Evaluating: What are the requirements for banks to reconcile their SGL a...
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Evaluation: ```
{
  "accuracy": 8,
  "completeness": 9,
  "clarity": 7,
  "relevance": 9,
  "overall_score": 8.25,
  "reasoning": "The generated answer accura

qa_id,question,category,difficulty,expected_answer,generated_answer,evaluation
afdcaad01b16aa9eb7a63a4576468b67723341a7c56dda016d845a44b15d276d,What forms of assets can be used for posting and collecting margin in India by Authorised Dealer Category-I banks for permitted derivative contracts?,fact-based,5,"Authorised Dealer Category-I (AD Cat-I) banks are permitted to post and collect margin in India for permitted derivative contracts using Indian currency, freely convertible foreign currency, debt securities issued by the Indian Central Government and State Governments, and Rupee bonds issued by persons resident in India that are listed on a recognized stock exchange in India and have a credit rating of AAA issued by a rating agency registered with the Securities and Exchange Board of India.","1. सरल विवरण: भारत में अधिकृत डीलर श्रेणी-1 बैंकों द्वारा अनुमति प्राप्त डेरिवेटिव अनुबंधों के लिए मार्जिन पोस्ट और इकट्ठा करने के लिए उपयोग किए जा सकने वाले परिसंपत्तियों के रूप में भारतीय मुद्रा, मुक्त रूपांतरणीय विदेशी मुद्रा, भारतीय केंद्र सरकार और राज्य सरकारों द्वारा जारी ऋण प्रतिभूतियां, और भारत में निवासी व्यक्तियों द्वारा जारी रुपये बॉन्ड शामिल हैं जो एक मान्यता प्राप्त स्टॉक एक्सचेंज में सूचीबद्ध हैं और जिन्हें सेबी के साथ पंजीकृत एक रेटिंग एजेंसी द्वारा एएए रेटिंग प्राप्त है। 2. यह किन पर लागू होता है: यह निर्देश सभी अधिकृत डीलर श्रेणी-1 बैंकों पर लागू होता है। 3. मुख्य कार्रवाई बिन्दु: - अधिकृत डीलर श्रेणी-1 बैंक भारत में अनुमति प्राप्त डेरिवेटिव अनुबंधों के लिए मार्जिन पोस्ट और इकट्ठा कर सकते हैं। - मार्जिन के रूप में भारतीय मुद्रा, मुक्त रूपांतरणीय विदेशी मुद्रा, भारतीय केंद्र सरकार और राज्य सरकारों द्वारा जारी ऋण प्रतिभूतियां, और रुपये बॉन्ड का उपयोग किया जा सकता है। - बैंकों को मार्जिन संबंधी लेन-देन के लिए अलग-अलग खाते रखने होंगे। 4. स्रोत चंक आईडी उपयोग किए गए: - c9d27e8924eb1f21b6ad6ee2e95a5883635a7182348a52e1511d14ddc281f040 - d49a76a90a22e9ad59d5968cab1fa5a917912c349a099a41bd6341e3e7024d69 - 89c1d2df89891043c24c3f69b9e2dde4456a0031a2e3b55cb6b62f39082f6231","``` { ""accuracy"": 9, ""completeness"": 8, ""clarity"": 7, ""relevance"": 9, ""overall_score"": 8.25, ""reasoning"": ""The generated answer accurately covers the key points from the expected answer, including the forms of assets that can be used for posting and collecting margin. However, it provides additional information not directly asked for, such as the applicability and main action points, which slightly reduces its clarity and completeness."" } ```"
9a9ee81924d7535d1404aae5f5134f7674f1ab80b559c5b04d010471bbe48add,"What are the requirements for banks to reconcile their SGL account balances with the Public Debt Office (PDO), and what reporting is required?",fact-based,5,"Banks are required to reconcile their SGL account balances with the balances in the books of the Public Debt Offices (PDOs) monthly. The reconciliation results should be presented to the Audit Committee of the Board, and the internal audit department should periodically check this reconciliation. Additionally, banks must send a quarterly certificate to the PDO confirming the SGL account balances have been reconciled and presented before the appropriate committee.","1. सरल विवरण: भारतीय रिज़र्व बैंक (आरबीआई) ने बैंकों के लिए अपने एसजीएल (सब्सक्राइबर्स गिल्ड लिमिटेड) अकाउंट बैलेंस को पब्लिक डेब्ट ऑफ़िस (पीडीओ) के साथ मिलाने के लिए दिशानिर्देश जारी किए हैं। बैंकों को अपने एसजीएल अकाउंट बैलेंस को पीडीओ के साथ मासिक रूप से मिलाना आवश्यक है, और यह मिलान उनके ऑडिट कमिटी के सामने रखा जाना चाहिए। इसके अलावा, बैंकों को पीडीओ को त्रैमासिक प्रमाण पत्र भी भेजना चाहिए, जिसमें यह पुष्टि की जाए कि उनके एसजीएल अकाउंट बैलेंस को मिलाना किया गया है। 2. यह किन पर लागू होता है: यह दिशानिर्देश सभी प्राथमिक (शहरी) सहकारी बैंकों पर लागू होते हैं। 3. मुख्य कार्रवाई बिन्दु: - बैंकों को अपने एसजीएल अकाउंट बैलेंस को पीडीओ के साथ मासिक रूप से मिलाना चाहिए। - मिलाने के परिणामों को ऑडिट कमिटी के सामने रखा जाना चाहिए। - बैंकों को पीडीओ को त्रैमासिक प्रमाण पत्र भेजने चाहिए, जिसमें एसजीएल अकाउंट बैलेंस के मिलाने की पुष्टि की